In [ ]:
"""
FePt Hemispherical Cap — Hysteresis Loop, Thickness Sweep (Pure L10, Static 0 K)
==================================================================================
Simulates hysteresis loops for a pure L10-phase FePt hemispherical cap on a
fixed 1 µm sphere, sweeping across a range of cap thicknesses. Both radial and
vertical uniaxial anisotropy modes are tested for each thickness.

Unlike the mixed-phase scripts, there is no A1 soft-phase fraction here —
every cell is assigned the full L10 anisotropy constant (Ku_hard). Solver is
MinDriver (static energy minimisation, 0 K). The SiO2 sphere is not included;
only the FePt shell is modelled.

Inputs  : Physical parameters defined in Section 1 (fixed diameter, thickness
          list, Ms, A, Ku, field range).
Outputs : One Excel file per (thickness, mode) combination, saved to a
          timestamped output folder.
"""

# --- Standard library ---
import os
from datetime import datetime

# --- Third-party ---
import numpy as np
import pandas as pd

# --- Micromagnetics stack ---
import micromagneticmodel as mm
import discretisedfield as df
import oommfc as oc


# ===========================================================================
# 1. PARAMETERS & STUDY SETUP
# ===========================================================================

# --- Material parameters (pure L10 FePt — no A1 soft phase) ---
Ms  = 1.0e6   # Saturation magnetisation [A/m]
A   = 1.0e-11 # Exchange stiffness [J/m]
Ku  = 6.6e6   # Uniaxial anisotropy constant — L10 phase only [J/m^3]

# --- Field sweep parameters ---
mu0    = 4 * np.pi * 1e-7  # Vacuum permeability [H/m]
B_max  = 18.0               # Maximum applied field magnitude [T]
B_tilt = 0.05               # Small in-plane tilt to break symmetry and avoid saddle points [T]

# --- Geometry: fixed sphere diameter, variable cap thickness ---
fixed_diameter = 1e-6  # Sphere diameter [m] — held constant throughout this study
thicknesses    = [10e-9, 20e-9, 40e-9, 60e-9, 80e-9]  # Cap thicknesses to sweep [m]

# --- Anisotropy modes to test per thickness ---
anisotropy_modes = [
    "Radial",           # Easy axis along the local surface normal (outward radial)
    "Uniaxial_Vertical" # Easy axis fixed along global z for all cells
]

# --- Field sweep: high → low → high (242 steps total) ---
B_fields = np.concatenate([
    np.linspace(B_max, -B_max, 121),
    np.linspace(-B_max, B_max, 121)
])

# --- Output ---
timestamp     = datetime.now().strftime("%Y-%m-%d_%H-%M")
OUTPUT_FOLDER = f'Janus_ThicknessSweep_1um_{timestamp}'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("=" * 80)
print(f"  THICKNESS SWEEP — Pure L10 FePt | 1 µm sphere | {timestamp}")
print("=" * 80)


# ===========================================================================
# 2. MAIN EXECUTION LOOP
# ===========================================================================

for t_val in thicknesses:
    for mode in anisotropy_modes:

        t_nm      = int(t_val * 1e9)  # Thickness in nm for labels
        safe_name = f"cap_1um_t{t_nm}nm_{mode}"
        excel_path = os.path.join(OUTPUT_FOLDER, f"results_{safe_name}.xlsx")

        # --- Resume support: skip if this combination already has saved data ---
        if os.path.exists(excel_path):
            print(f"[*] Skipping {safe_name}: file already exists.")
            continue

        # --- Geometry: hemispherical shell radii ---
        R_inner = fixed_diameter / 2
        R_outer = R_inner + t_val

        # Cell size adapts to thickness: thin films need finer cells to resolve
        # the shell properly; thicker films can afford coarser discretisation
        cell_size = 5e-9 if t_val <= 20e-9 else 10e-9  # [m]

        # Non-cubic mesh: full width in x/y, half-height in z (upper hemisphere only)
        nx = int(round(2 * R_outer / cell_size))
        ny = int(round(2 * R_outer / cell_size))
        nz = int(round(R_outer / cell_size))       # z goes from 0 to R_outer only

        # Upper half-space only (z >= 0) — models just the cap, not the full sphere
        region = df.Region(p1=(-R_outer, -R_outer, 0),
                           p2=( R_outer,  R_outer, R_outer))
        mesh = df.Mesh(region=region, n=(nx, ny, nz))

        print(f"\n[RUNNING] Thickness: {t_nm} nm | Mode: {mode} | "
              f"Cell: {cell_size*1e9:.0f} nm | Mesh: ({nx}, {ny}, {nz})")

        # --- Spatial field functions assigned cell-by-cell ---

        def Ms_mask(pos):
            """Return Ms inside the hemispherical shell, 0 elsewhere."""
            r = np.sqrt(pos[0]**2 + pos[1]**2 + pos[2]**2)
            return Ms if (R_inner <= r <= R_outer and pos[2] >= 0) else 0

        def u_func(pos):
            """
            Easy-axis direction per cell.
            Radial: axis points along the local surface normal (outward radial).
            Uniaxial_Vertical: axis is fixed along global z for every cell.
            Falls back to (0,0,1) at the origin to avoid division by zero.
            """
            if mode == "Radial":
                r = np.linalg.norm(pos)
                return (pos[0]/r, pos[1]/r, pos[2]/r) if r != 0 else (0, 0, 1)
            return (0, 0, 1)  # Uniaxial_Vertical

        # --- Build micromagnetic system ---
        system = mm.System(name=safe_name)
        system.energy = (
            mm.Exchange(A=A)
            + mm.Demag()
            + mm.UniaxialAnisotropy(
                K=Ku,  # Uniform L10 anisotropy — no spatial distribution needed
                u=df.Field(mesh, nvdim=3, value=u_func)
            )
            + mm.Zeeman(H=(0, 0, 0))  # Field updated at each step below
        )

        # Initialise magnetisation uniformly along +z
        system.m = df.Field(mesh, nvdim=3, value=(0, 0, 1),
                            norm=df.Field(mesh, nvdim=1, value=Ms_mask))

        # --- Field sweep with energy minimisation at each step ---
        hyst_data = []
        md = oc.MinDriver()  # Static minimiser — no dynamics, effectively 0 K

        try:
            for s_idx, B_z in enumerate(B_fields):
                # Small By tilt prevents the solver getting stuck at a saddle point
                system.energy.zeeman.H = (0, B_tilt / mu0, B_z / mu0)
                md.drive(system, n_threads=8, stopping_mxHxm=2e4)

                # Normalised average Mz over magnetic material only
                # .item() converts the scalar field integral to a plain Python float
                total_mz     = system.m.z.integrate()
                total_ms_vol = system.m.norm.integrate()
                mz_material_avg = (total_mz / total_ms_vol).item()

                hyst_data.append({'B_ext (T)': B_z, 'M_z/Ms': mz_material_avg})

                # Progress log every 40 steps
                if s_idx % 40 == 0:
                    print(f"  Step {s_idx+1}/242 | B: {B_z:6.2f} T | Mz/Ms: {mz_material_avg:8.4f}")

            # Save this (thickness, mode) combination to its own Excel file
            pd.DataFrame(hyst_data).to_excel(excel_path, index=False)
            print(f"[+] Saved: {excel_path}")

        except Exception as e:
            print(f"[!] Error in {safe_name}: {e}")
            continue  # Move on to the next combination rather than crashing the whole sweep

        oc.delete(system)    # Free OOMMF system object
        oc.Runner().clear()  # Clear any lingering OOMMF runner state


# ===========================================================================
# 3. COMPLETION SUMMARY
# ===========================================================================

print(f"\nThickness sweep complete. Results saved in: {OUTPUT_FOLDER}")
